# SPEC2RTL & PROMPT2SILICON: CĂN CHỈNH DPO & XUẤT GGUF
### Kế hoạch Chưng cất + Căn chỉnh Phần cứng (Hardware Preference Alignment)
**Nhóm 2:** Huấn luyện AI Cục bộ & Tinh chỉnh Nâng cao (DPO + GGUF Export)

- **Chế độ dữ liệu mặc định (`USE_PURE_TEACHER_DISTILL = True`):** **1,794 cặp `(chosen vs rejected)`** gồm **157 cặp thật** từ Teacher Self-Correction + đột biến 3 lỗi phần cứng (Latch, Dangling Pin, Blocking Race Condition).
- **Chế độ tập gộp (`USE_PURE_TEACHER_DISTILL = False`):** **2,881 cặp** từ toàn bộ tập SFT hợp nhất.
- *Khuyên dùng:* Chạy notebook này ngay trong cùng phiên Colab sau khi chạy xong `train_spec2rtl_qwen3b_colab.ipynb` để DPO học nối tiếp trên `spec2rtl_sft_lora` (hoặc upload `spec2rtl_sft_lora.zip` lên Colab).

## Bước 1: Kiểm tra phần cứng GPU T4 Google Colab
*Vào menu **Runtime -> Change runtime type -> T4 GPU** (miễn phí).*

In [ ]:
!nvidia-smi
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Bước 2: Cài đặt Unsloth & Thư viện Huấn luyện DPO (TRL)

In [ ]:
%%capture
!pip install -q unsloth matplotlib

## Bước 3: Nạp mô hình SFT LoRA (hoặc mô hình nền nếu chạy độc lập)

In [ ]:
import os, zipfile, torch
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()

# Phải khớp MODEL_SIZE với bước SFT ("3B" mặc định; đổi thành "0.5B"/"1.5B" nếu máy ít RAM)
MODEL_SIZE = "3B"
MODEL_MAP = {
    "0.5B": "unsloth/Qwen2.5-Coder-0.5B-Instruct-bnb-4bit",
    "1.5B": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "3B":   "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
}

max_seq_length = 2048
sft_dir = "spec2rtl_sft_lora"
if not os.path.exists(sft_dir) and os.path.exists("spec2rtl_sft_lora.zip"):
    print("[*] Giải nén spec2rtl_sft_lora.zip...")
    with zipfile.ZipFile("spec2rtl_sft_lora.zip", "r") as zf:
        zf.extractall(".")

if os.path.exists(sft_dir):
    print(f"[*] Phát hiện trọng số SFT từ bước trước: {sft_dir} -> Tiếp tục căn chỉnh DPO trên SFT!")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = sft_dir,
        max_seq_length = max_seq_length,
        load_in_4bit = True,
        dtype = None,
    )
else:
    model_name = MODEL_MAP[MODEL_SIZE]
    print(f"[*] Không thấy {sft_dir}, nạp mô hình nền {model_name} + gắn LoRA mới cho DPO...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        load_in_4bit = True,
        dtype = None,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
print(">>> Đã sẵn sàng mô hình cho DPO Alignment!")

## Bước 4: Tự động tải Tập dữ liệu DPO Căn chỉnh Phần cứng từ GitHub (`LamGianggg/TKLLS`)

In [ ]:
import urllib.request
import json
from datasets import Dataset

# True = 1,794 cặp từ tập thuần Gemini Teacher + 157 cặp Self-Correction; False = 2,881 cặp tập gộp đầy đủ
USE_PURE_TEACHER_DISTILL = True
DPO_FILE = "dataset_teacher_dpo.json" if USE_PURE_TEACHER_DISTILL else "dataset_dpo_preference.json"
DPO_URL = f"https://raw.githubusercontent.com/LamGianggg/TKLLS/main/training/{DPO_FILE}"
print(f"[*] Đang tải dữ liệu DPO từ: {DPO_URL}...")
urllib.request.urlretrieve(DPO_URL, DPO_FILE)

with open(DPO_FILE, "r", encoding="utf-8") as f:
    raw_dpo_data = json.load(f)

print(f">>> Đã nạp thành công {len(raw_dpo_data)} cặp dữ liệu DPO (Chosen vs Rejected)!")

formatted_data = {
    "prompt": [],
    "chosen": [],
    "rejected": []
}

system_prompt = (
    "Bạn là chuyên gia thiết kế vi mạch RTL và Verilog synthesizable chuẩn công nghiệp IEEE 1364-2001. "
    "Luôn sinh mã chuẩn sạch, đầy đủ cổng, không tạo chốt ẩn (no latches) và đúng quy tắc gán tuần tự/tổ hợp."
)

for item in raw_dpo_data:
    p_text = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{item['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    c_text = f"{item['chosen']}<|im_end|>"
    r_text = f"{item['rejected']}<|im_end|>"

    formatted_data["prompt"].append(p_text)
    formatted_data["chosen"].append(c_text)
    formatted_data["rejected"].append(r_text)

dpo_dataset = Dataset.from_dict(formatted_data)
print(f">>> Dataset DPO đã sẵn sàng: {len(dpo_dataset)} cặp!")
print("Ví dụ mẫu 0 (Lý do Rejected):", raw_dpo_data[0].get("rejected_error_type", "N/A"))

## Bước 5: Cấu hình và Chạy Huấn luyện DPO (Direct Preference Optimization)
Mô hình học tối đa hóa xác suất chọn `chosen` (code chuẩn qua ModelSim) và phạt `rejected` (code có chốt ẩn, chân lơ lửng, race condition).

In [ ]:
import inspect
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

cfg_sig = inspect.signature(DPOConfig.__init__).parameters
cfg_kwargs = dict(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.1,
    num_train_epochs = 1,
    learning_rate = 5e-6,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 10,
    save_strategy = "no",
    optim = "adamw_8bit",
    weight_decay = 0.01,
    beta = 0.1,
    seed = 3407,
    output_dir = "dpo_outputs",
    report_to = "none",
)
if "max_prompt_length" in cfg_sig:
    cfg_kwargs["max_prompt_length"] = 768
if "max_length" in cfg_sig:
    cfg_kwargs["max_length"] = 2048

dpo_config = DPOConfig(**cfg_kwargs)

trainer_sig = inspect.signature(DPOTrainer.__init__).parameters
trainer_kwargs = dict(
    model = model,
    ref_model = None,
    args = dpo_config,
    train_dataset = dpo_dataset,
)
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_sig:
    trainer_kwargs["tokenizer"] = tokenizer
if "beta" in trainer_sig and "beta" not in cfg_kwargs:
    trainer_kwargs["beta"] = 0.1
if "max_length" in trainer_sig and "max_length" not in cfg_kwargs:
    trainer_kwargs["max_length"] = 2048
if "max_prompt_length" in trainer_sig and "max_prompt_length" not in cfg_kwargs:
    trainer_kwargs["max_prompt_length"] = 768

dpo_trainer = DPOTrainer(**trainer_kwargs)

print(">>> BẮT ĐẦU HUẤN LUYỆN DPO (CĂN CHỈNH PHẦN CỨNG)...")
dpo_stats = dpo_trainer.train()
print(">>> HUẤN LUYỆN DPO THÀNH CÔNG RỰC RỠ!")

## Bước 5.5: Vẽ Biểu Đồ Hội Tụ DPO Loss (Lưu ảnh cho Báo Cáo / Slide)

In [ ]:
import matplotlib.pyplot as plt

history = dpo_trainer.state.log_history
steps = [x["step"] for x in history if "loss" in x]
losses = [x["loss"] for x in history if "loss" in x]

if steps and losses:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, marker="o", markersize=3, color="#2563eb", linewidth=2, label="DPO Loss")
    plt.title("Biểu Đồ Hội Tụ DPO Loss (Hardware Preference Alignment)", fontsize=14, fontweight="bold")
    plt.xlabel("Số bước huấn luyện (Training Steps)", fontsize=12)
    plt.ylabel("Hàm mất mát (DPO Loss)", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig("dpo_loss_convergence.png", dpi=300)
    plt.show()
    print(">>> Đã lưu ảnh biểu đồ: dpo_loss_convergence.png (Tải về bỏ vào docs/dpo_loss_convergence.png)")
else:
    print("Chưa có lịch sử loss để vẽ đồ thị.")

## Bước 6: Kiểm tra Suy luận Sau khi Căn chỉnh DPO

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """Hãy thiết kế module Verilog synthesizable cho mạch khử rung nút bấm: button_debouncer

[Đặc tả cổng]:
- Inputs: clk (xung nhịp), rst (reset tích cực cao), btn_in (tín hiệu vật lý từ nút bấm cơ học).
- Outputs: btn_clean (tín hiệu nút sạch sau khi khử rung), btn_pressed_pulse (xung 1 chu kỳ khi nhấn).

[Yêu cầu]:
- Tuyệt đối không để chân btn_in bị lơ lửng (dangling pin).
- Lấy mẫu btn_in qua thanh ghi dịch hoặc bộ đếm thời gian ổn định."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

from transformers import TextStreamer
streamer = TextStreamer(tokenizer)
print("=== KẾT QUẢ MÔ HÌNH SINH CODE SAU DPO ===")
_ = model.generate(input_ids = inputs, streamer = streamer, max_new_tokens = 768, use_cache = True)

## Bước 7: Lưu LoRA Adapter & Xuất file GGUF (`q4_k_m`) để chạy offline trên máy local

In [ ]:
# 1. Lưu LoRA adapter DPO đúng tên thư mục local (models/qwen2.5_3b_verilog_dpo)
dpo_dir = f"qwen2.5_{MODEL_SIZE.lower()}_verilog_dpo"
model.save_pretrained(dpo_dir)
tokenizer.save_pretrained(dpo_dir)
!zip -qr {dpo_dir}.zip {dpo_dir}
print(f">>> Đã lưu và nén DPO LoRA adapter: {dpo_dir}.zip")

# 2. Xuất file GGUF Q4_K_M (~2.0 GB cho bản 3B) để chạy offline bằng models/run_inference.py
gguf_dir = f"spec2rtl_{MODEL_SIZE.lower()}_dpo_gguf"
print(f"[*] Đang hợp nhất và lượng tử hóa sang GGUF (q4_k_m) vào: {gguf_dir}...")
model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method = "q4_k_m")
!ls -lh {gguf_dir}/*.gguf
print(">>> HOÀN TẤT! Tải file .gguf và dpo_loss_convergence.png về máy bỏ vào models/ và docs/!")